In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from utils import (
    load_task_incremental_from_pretrain,
    load_task_incremental_classifier,
    save_task_incremental_classifier,
 )
import torch.nn.functional as F
import torch
from train import train_classifier, evaluate_task_incremental
from dataloaders import SequentialCIFAR10
from torch import nn

BATCH_SIZE = 128
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
TASK_IL_CKPT_DIR = "checkpoints/task_0_pretrain"
LWF_CKPT_DIR = "checkpoints/lwf_task_il"
seq_cifar = SequentialCIFAR10(data_root="./data", batch_size=BATCH_SIZE, buffer_size=200)

In [7]:
# calcular etiquetas de modelo viejo sobre dataset nuevo

def calculate_old_model_labels(model, task_id, dataloader):
    model.eval()
    old_model_labels = []
    with torch.no_grad():
        for inputs, _ in dataloader:
            inputs = inputs.to(device)
            outputs = model.forward_probs(inputs, task_id)  # Asumiendo que el modelo puede manejar entradas sin task_id
            old_model_labels.append(outputs.cpu())
    return torch.cat(old_model_labels).to(device)



In [ ]:
class LWFLoss(nn.Module):

    def __init__(self,  old_tasks_labels:list[torch.Tensor], old_task_ids, peso_old_task=0.5, T= 2, weight_decay=1e-4,):
        super(LWFLoss, self).__init__()
        self.old_task_labels_reescalado =[self.suavizar_labels(labels, T) for labels in old_tasks_labels]
        self.old_task_ids = old_task_ids
        self.peso_old_task = peso_old_task
        self.T = T
        self.weight_decay = weight_decay
        self.ce_loss = nn.CrossEntropyLoss()
    def suavizar_labels(self, labels, T):
        return labels**(1/T) / torch.sum(labels**(1/T), dim=1, keepdim=True)
    def forward(self, images, labels, current_task_id, model, batch_size, batch_idx):
        outputs = model(images, current_task_id)
        ce_loss = self.ce_loss(outputs, labels)

        weight_decay_term = 0.0
        for param in model.parameters():
            weight_decay_term += torch.sum(param ** 2)
        weight_decay_term *= self.weight_decay

        distillation_loss = 0.0
        sample_start_idx = batch_idx * batch_size
        for task_id in self.old_task_ids:
            old_task_output = self.suavizar_labels(model.forward_probs(images, task_id=task_id), self.T)
            distillation_loss = - torch.sum(self.old_task_labels_reescalado[task_id][sample_start_idx: sample_start_idx + batch_size] 
                                            * torch.log(old_task_output))
            
        return   ce_loss + (self.peso_old_task) * distillation_loss + weight_decay_term
def train_lwf_classifier(old_model, task_id, train_dataloader, valid_dataloader, optimizer, warmup_epochs=5,
                         joint_optimization_epochs = 5, 
                         alpha=0.5):
    old_model.to(device)
    old_model.add_task(task_id, num_classes=2)

    old_task_ids = [i for i in old_model.task_ids() if i != task_id]
    old_tasks_labels = [calculate_old_model_labels(old_model, old_task_id, train_dataloader) for old_task_id in old_task_ids]
    loss_function = LWFLoss(old_tasks_labels, old_task_ids, peso_old_task=0.5, T=2, weight_decay=1e-4)

    train_losses = []
    old_model.freeze_backbone()
    for epoch in range(warmup_epochs):
        cum_loss = 0
        for batch_idx, (inputs, labels) in enumerate(train_dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            # Warmup NO NECESITA EL TERMINO DE DISTILLATION PORQUE SOLO SE ENTRENAN LOS NUEVOS HEADS, LOS PARAMETROS COMPARTIDOS ESTAN CONGELADOS

            loss = loss_function(inputs, labels, task_id, old_model, BATCH_SIZE, batch_idx)
            loss.backward()
            cum_loss += loss.item()
            optimizer.step()
        avg_loss = cum_loss / len(train_dataloader)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch + 1}/{warmup_epochs}, Loss: {avg_loss:.4f}")

    # Joint training
    old_model.unfreeze_backbone()
    for epoch in range(joint_optimization_epochs):
        cum_loss = 0
        for batch_idx, (inputs, labels) in enumerate(train_dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_function(inputs, labels, task_id, old_model, BATCH_SIZE, batch_idx)
            loss.backward()
            cum_loss += loss.item()
            optimizer.step()
        avg_loss = cum_loss / len(train_dataloader)
        train_losses.append(avg_loss)
        print(f"Joint Optimization Epoch {epoch + 1}/{joint_optimization_epochs}, Loss: {avg_loss:.4f}")

In [9]:
first_model = load_task_incremental_from_pretrain(TASK_IL_CKPT_DIR, task_id=0, device = torch.device("mps"))

Backbone + task head loaded successfully into task-incremental classifier.


In [ ]:
train_loader, val_loader = seq_cifar.get_task_il_train_val_loaders(task_id=1)
train_lwf_classifier(
    old_model=first_model,
    task_id=1,
    train_dataloader=train_loader,
    valid_dataloader=val_loader,
    optimizer=torch.optim.SGD(first_model.parameters(), lr=0.01, momentum=0.9),
    warmup_epochs=2,
    joint_optimization_epochs = 5,
    alpha=0.5
)

/Users/alexanderbodner/Documents/Udesa/5to/vision avanzada/tp1_continual_learning/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


45.120296478271484
45.15608596801758
45.150718688964844
45.15378189086914
45.08197021484375
45.17744064331055
45.13323211669922
45.119022369384766
45.15223693847656
45.08491134643555
45.15134048461914
45.118770599365234
45.193634033203125
45.188621520996094
45.11876678466797
45.155147552490234
45.138397216796875
45.12942886352539
45.160213470458984
45.09421157836914
45.13621520996094
45.13515853881836
45.126502990722656
45.125160217285156
45.13364028930664
45.1438102722168
45.15744400024414
45.112579345703125
45.12149429321289
45.0987434387207
45.10478591918945
45.159610748291016
45.12911605834961
45.140472412109375
45.129058837890625
45.138153076171875
45.18864059448242
45.178401947021484
45.14035415649414
45.16339111328125
45.15748596191406
45.14596939086914
45.13313293457031
45.10069274902344
45.12321472167969
45.15791702270508
45.124847412109375
45.1119270324707
45.12989044189453
45.132728576660156
45.13787841796875
45.1314582824707
45.111289978027344
45.101219177246094
45.13490676